# Gemma 4 Developer Agent — Full Harness Edition

**Goal:** maximize hidden-test resolution by configuring the actual competition harness: `swegemma` + `adk-submission` + `adk-eval-core`, Google ADK, vLLM/LiteLLM, context cache/compaction, repository tools, skills, sandbox patch extraction and Phase-2 verification.

The uploaded baseline smoke-tested only 2 of 129 public tasks; both returned `NO_PATCH` (0/2) with one-minute budgets. That is not a reliable leaderboard estimate. Community experiments report roughly 47–49/129 (36–38%) on the public development set, so 1.0 is an ambitious target, not a measured or guaranteed outcome. A public-set 1.0 would still not prove 1.0 on hidden private repositories.

The default is Gemma 4 31B QAT. Official competition guidance also lists 12B, E4B and E2B QAT variants; select exactly one model for every agent. The notebook evaluates the full 129-task public split by default, writes crash-safe per-task checkpoints, and stops locally before the 12-hour aggregate ceiling.

## 1. Offline runtime setup

Attach the competition dataset, official offline wheelhouse, and selected model input. Internet access stays disabled. The global safety timer starts before installation and model startup.

In [ ]:
import glob, importlib, hashlib, json, logging, os, re, shutil, subprocess, sys, time, zipfile
from collections import Counter
from pathlib import Path

SESSION_STARTED = time.monotonic()
os.environ.update({
    "TRANSFORMERS_NO_TF":"1", "TRANSFORMERS_NO_FLAX":"1", "TRANSFORMERS_OFFLINE":"1",
    "VLLM_NO_USAGE_STATS":"1", "VLLM_WORKER_MULTIPROC_METHOD":"spawn",
    "VLLM_ENGINE_READY_TIMEOUT_S":"1200", "NCCL_DEBUG":"WARN",
    "LITELLM_TELEMETRY":"False", "LITELLM_LOG":"WARNING",
    "OTEL_SDK_DISABLED":"true", "OTEL_PYTHON_DISABLED":"true",
    "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
    "OPENAI_API_KEY":os.environ.get("OPENAI_API_KEY","EMPTY"),
    "LOCAL_API_KEY":os.environ.get("LOCAL_API_KEY","EMPTY"),
})
for name in ("litellm","LiteLLM","httpx","httpcore","openai","urllib3","opentelemetry"):
    logging.getLogger(name).setLevel(logging.WARNING)

DATA_DIR=Path("/kaggle/input/competitions/gemma-4-developer-agent")
WHEELHOUSE=Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
WORK=Path("/kaggle/working"); WORK.mkdir(parents=True,exist_ok=True)
for p in (DATA_DIR/"tasks.jsonl",DATA_DIR/"sample_submission",WHEELHOUSE):
    if not p.exists(): raise FileNotFoundError(f"Missing Kaggle input: {p}")

# Repair CUDA wheel tags if '+' was stripped from cu128 filenames.
tmp_whl=Path("/tmp/gemma4-wheelhouse"); tmp_whl.mkdir(parents=True,exist_ok=True)
for wheel in WHEELHOUSE.glob("*.whl"):
    if "cutlass" in wheel.name.lower(): continue
    name=wheel.name.replace("cu128","+cu128") if "cu128" in wheel.name and "+" not in wheel.name else wheel.name
    target=tmp_whl/name
    if not target.exists():
        try: os.symlink(wheel,target)
        except OSError: shutil.copyfile(wheel,target)
wheels=sorted(str(p) for p in tmp_whl.glob("*.whl"))
if not wheels: raise RuntimeError("No compatible offline wheels found.")
for pattern in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth","/usr/local/lib/python*/site-packages/*cutlass*.pth"):
    for p in glob.glob(pattern):
        try: os.unlink(p)
        except OSError: pass
subprocess.run([sys.executable,"-m","pip","install","-q","--no-deps","--force-reinstall",*wheels],check=True)
importlib.invalidate_caches()
print(f"Offline runtime prepared; installed {len(wheels)} wheels.")

## 2. Configure the model, root coder, analyzer and per-task policy

The root agent owns edits and patch handoff. The read-only analyzer is an `agent_tool` with `skip_summarization: true`, used only when localization stalls. Skills give deterministic navigation and a final patch hygiene check.

In [ ]:
from swegemma.models import load_tasks

TASKS_PATH=DATA_DIR/"tasks.jsonl"
tasks=load_tasks(TASKS_PATH)
if not tasks: raise RuntimeError("No public development tasks loaded.")
counts=Counter(t.repo for t in tasks)
print(f"Loaded {len(tasks)} tasks across {len(counts)} repositories: {dict(counts)}")

ALLOWED_MODELS={
    "gemma-4-31b-it-qat-w4a16-ct","gemma-4-12b-it-qat-w4a16-ct",
    "gemma-4-e4b-it-qat-w4a16-ct","gemma-4-e2b-it-qat-w4a16-ct",
}
MODEL_NAME=os.environ.get("GEMMA_MODEL_NAME","gemma-4-31b-it-qat-w4a16-ct").strip()
if MODEL_NAME not in ALLOWED_MODELS: raise ValueError(f"Unsupported model: {MODEL_NAME}")

AGENT_DIR=WORK/"gemma4_agent"
if AGENT_DIR.exists(): shutil.rmtree(AGENT_DIR)
shutil.copytree(DATA_DIR/"sample_submission",AGENT_DIR)

# Starter LoRA files are tiny placeholders, not validated fine-tunes. Strip by default.
USE_VALIDATED_LORA=os.environ.get("USE_VALIDATED_LORA","0").lower() in {"1","true","yes"}
yaml_paths=[AGENT_DIR/"agent.yaml",*sorted((AGENT_DIR/"sub_agents").glob("*.yaml"))]
for y in yaml_paths:
    text=y.read_text(encoding="utf-8")
    text=re.sub(r"(?m)^([ \t]*model:[ \t]*).+$",rf"\g<1>{MODEL_NAME}",text)
    if not USE_VALIDATED_LORA: text=re.sub(r"(?m)^[ \t]*adapter:[^\n]*\n?","",text)
    y.write_text(text,encoding="utf-8")
if not USE_VALIDATED_LORA:
    shutil.rmtree(AGENT_DIR/"adapters",ignore_errors=True)
else:
    configs=list((AGENT_DIR/"adapters").glob("*/adapter_config.json"))
    if not configs: raise FileNotFoundError("No adapter configs supplied.")
    for cfg in configs:
        meta=json.loads(cfg.read_text(encoding="utf-8"))
        if MODEL_NAME.lower() not in str(meta.get("base_model_name_or_path","")).lower():
            raise ValueError(f"Adapter/base model mismatch: {cfg}")
        if not (cfg.parent/"adapter_model.safetensors").is_file():
            raise FileNotFoundError(f"Adapter weights missing: {cfg.parent}")

SYSTEM_PROMPT=r'''
You are the root autonomous software engineer in the SWE-Gemma harness. Repair the issue in /workspace with the smallest correct source patch that passes its validation tests.

RULES
- Treat the complete problem statement and hints as the specification. Extract behavior, affected APIs, error messages, compatibility constraints and edge cases.
- Work only in /workspace. The environment is air-gapped; use preinstalled dependencies and /wheels only when demonstrably needed.
- Modify implementation/source only. Never add, edit, delete or weaken tests, fixtures, snapshots, pytest configuration, conftest.py or harness files. The evaluator protects/resets these.
- Preserve public interfaces, exception types and exact error wording unless the task explicitly requires a change. Never fabricate output, source locations, graph results or test outcomes.

ORDERED REPAIR LOOP
1. PARSE: identify task nouns, identifiers, filenames, stack frames and required behavior. Form 2–5 likely edit locations.
2. LOCALIZE: read named paths/symbols directly. Otherwise run one narrow, high-signal rg/grep search. Inspect implementation and callers before editing.
3. GRAPH: graph tools are indexed by symbols. Obtain an exact dotted node ID from source/grep first; then use get_code_neighbors. Use subgraph only for a small, relevant cluster. Treat source as authoritative.
4. ANALYZER: use read-only code_analyzer at most once, only if focused inspection leaves real ambiguity or the fix spans modules. Give it exact symbols already found.
5. REPRODUCE: when useful, put scratch code in /tmp, never /workspace; submit_patch captures untracked workspace files.
6. PATCH: make the smallest root-cause change with repository helpers. Read exact old text before edit_file; ensure unique replacement. Avoid unrelated refactors and dependencies.
7. VERIFY: run the narrowest existing test module/test name or a targeted assertion. Never run bare pytest or the full suite. Inspect the first meaningful failure and allow at most two focused correction cycles.
8. HYGIENE: run patch_drafting/scripts/diff_guard.py through run_skill_script. Inspect status, git diff --check and git diff --stat. Never git add -A, commit or stash.
9. SUBMIT: explicitly call submit_patch() once after verification/hygiene. It is free and required. Do not end with only a verbal summary.

BUDGET
Maximum 50 paid tool calls, 60 reasoning turns, 5 minutes per task. Reserve at least four paid calls for final verification. get_status and submit_patch are free. Never repeat identical failed calls; change localization method if evidence stalls. Always submit; never invent an unrelated edit just to create a patch.
'''.strip()
ANALYZER_PROMPT=r'''
You are a READ-ONLY source-localization specialist. Do not edit, test or submit.
Inspect only the smallest source set relevant to the task. Use graph tools only with exact dotted symbol IDs. Return up to five ranked candidate edit sites with file/symbol, line evidence, supported caller/callee links, facts vs hypotheses, nearest existing pattern and compatibility risks. State what evidence is missing; do not propose broad refactors.
'''.strip()

(AGENT_DIR/"prompts").mkdir(parents=True,exist_ok=True)
(AGENT_DIR/"prompts/system.md").write_text(SYSTEM_PROMPT+"\n",encoding="utf-8")
(AGENT_DIR/"prompts/analyzer.md").write_text(ANALYZER_PROMPT+"\n",encoding="utf-8")
(AGENT_DIR/"configs/sampling.yaml").write_text(
    "temperature: 0.6\ntop_p: 0.95\ntop_k: 64\nmax_output_tokens: 8192\n"
    "thinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n",encoding="utf-8")
(AGENT_DIR/"eval_config.yaml").write_text(
    "# max_time_minutes is per task; total competition cap is 12 hours.\n"
    "evaluation:\n  timeout_seconds: 300\n  max_tool_calls: 50\n  max_turns: 60\n  max_time_minutes: 5\n",encoding="utf-8")

(AGENT_DIR/"agent.yaml").write_text(f'''name: swe_gemma_root_coder
model: {MODEL_NAME}
instruction: !include prompts/system.md
generate_content_config: !include configs/sampling.yaml
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
  - run_skill_script
  - load_skill_resource
  - agent_tool:
      config_path: sub_agents/code_analyzer.yaml
      skip_summarization: true
skills:
  - skills/repo_navigation
  - skills/patch_drafting
''',encoding="utf-8")
(AGENT_DIR/"sub_agents/code_analyzer.yaml").write_text(f'''name: code_analyzer_agent
description: Read-only repository-localization specialist, called only when root-coder localization stalls.
model: {MODEL_NAME}
instruction: !include ../prompts/analyzer.md
generate_content_config: !include ../configs/sampling.yaml
tools:
  - run_command
  - read_file
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
''',encoding="utf-8")
print(f"Prepared single-model agent {MODEL_NAME}; validated LoRA={USE_VALIDATED_LORA}")

In [ ]:
# Skills are bounded deterministic helpers; they do not introduce a second writer agent.
repo_skill=AGENT_DIR/"skills/repo_navigation"
patch_skill=AGENT_DIR/"skills/patch_drafting"
for p in (repo_skill/"scripts",repo_skill/"resources",patch_skill/"scripts",patch_skill/"resources"):
    p.mkdir(parents=True,exist_ok=True)
(repo_skill/"SKILL.md").write_text("""---
name: repo_navigation
description: Bounded Python-symbol localization when issue paths are unclear.
---
Use scripts/find_symbol.py with a distinctive expression only when the issue lacks a concrete path or symbol.
""",encoding="utf-8")
(repo_skill/"scripts/find_symbol.py").write_text(r'''#!/usr/bin/env python3
import os,re,sys
pat=re.compile(sys.argv[1] if len(sys.argv)>1 else r"^\s*(?:async\s+)?def\s+\w+|^\s*class\s+\w+")
limit=max(1,min(int(sys.argv[2]) if len(sys.argv)>2 else 40,100))
skip={".git",".venv","venv","__pycache__","node_modules","build","dist"}
hits=0
for base,dirs,files in os.walk("."):
    dirs[:]=[d for d in dirs if d not in skip]
    for name in sorted(files):
        if not name.endswith(".py"): continue
        path=os.path.join(base,name)
        try:
            with open(path,encoding="utf-8",errors="replace") as stream:
                for line_no,line in enumerate(stream,1):
                    if pat.search(line):
                        print(f"{path}:{line_no}: {line.strip()[:150]}")
                        hits+=1
                        if hits>=limit: raise SystemExit(0)
        except OSError: pass
print(f"TOTAL_HITS={hits}")
''',encoding="utf-8")
(repo_skill/"resources/localization.md").write_text(
    "Start from task identifiers, error strings and the deepest repository traceback frame. "
    "Graph tools need exact IDs copied from grep/source. Inspect callers and nearby tests; never edit tests.",encoding="utf-8")
(patch_skill/"SKILL.md").write_text("""---
name: patch_drafting
description: Pre-submit source-only diff and protected-path hygiene check.
---
Run scripts/diff_guard.py immediately before submit_patch and fix every reported problem.
""",encoding="utf-8")
(patch_skill/"scripts/diff_guard.py").write_text(r'''#!/usr/bin/env python3
import subprocess,sys
def run(args):
    p=subprocess.run(args,text=True,capture_output=True,timeout=20)
    if p.returncode: raise RuntimeError(p.stderr.strip() or f"command failed: {args}")
    return p.stdout.strip()
# Match submit_patch's intent-to-add behavior for new implementation files.
run(["git","add","-N","."])
changed=run(["git","diff","--name-only","HEAD"]).splitlines()
protected={"conftest.py","pytest.ini","pyproject.toml","tox.ini","setup.cfg",
           ".pytest.ini","sitecustomize.py","usercustomize.py","_swegemma_stubs.py"}
problems=[]
for name in changed:
    parts=name.lower().split("/")
    leaf=parts[-1]
    if leaf in protected or leaf.endswith(".pth") or any(x in {"tests","test","testing"} for x in parts[:-1]) or leaf.startswith("test_") or leaf.endswith("_test.py"):
        problems.append("PROTECTED TEST/CONFIG CHANGE: "+name)
untracked=run(["git","ls-files","--others","--exclude-standard"])
if untracked: problems.append("SCRATCH/UNTRACKED FILE IN /workspace:\n"+untracked)
if not changed: problems.append("EMPTY DIFF: verify whether an implementation edit is justified.")
if len(changed)>8: problems.append("MORE THAN 8 FILES CHANGED: remove unrelated work.")
run(["git","diff","--check","HEAD"])
if problems:
    print("\n".join(problems)); sys.exit(2)
print("PATCH_GUARD_OK files="+str(len(changed)))
print(run(["git","diff","--stat","HEAD"]))
''',encoding="utf-8")
(patch_skill/"resources/checklist.md").write_text(
    "Change only requested implementation files. Preserve APIs, exception types and error text. "
    "Never edit tests/fixtures/config. Keep scratch in /tmp. Run a targeted test. Check diff and submit.",encoding="utf-8")

# Enforce shared model and safe relative !include paths before compilation.
for y in (AGENT_DIR/"agent.yaml",AGENT_DIR/"sub_agents/code_analyzer.yaml"):
    text=y.read_text(encoding="utf-8")
    declared=set(re.findall(r"(?m)^\s*model:\s*(\S+)",text))
    if declared and declared!={MODEL_NAME}: raise AssertionError(f"Model mismatch in {y}: {declared}")
    for include in re.findall(r"!include\s+(\S+)",text):
        target=(y.parent/include).resolve()
        if not target.is_file() or AGENT_DIR.resolve() not in target.parents:
            raise FileNotFoundError(f"Invalid !include {include} in {y}")
print("Agent tree, skills, model declarations and !include paths preflighted.")

## 3. Preflight and package the submission

The local validation uses the competition's official `adk_submission` validator, then checks symlinks, supported extensions, the 3 GiB unpacked limit, ZIP CRC, and root-level `agent.yaml`. The archive is built before the expensive evaluation.

In [ ]:
from adk_submission import validate_directory
from swegemma.config import (
    ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES, build_submission_limits,
)
limits,generation_constraints=build_submission_limits()
validate_directory(AGENT_DIR,limits=limits)
unpacked_bytes=0
for base,dirs,files in os.walk(AGENT_DIR,followlinks=False):
    for name in [*dirs,*files]:
        p=Path(base)/name
        if p.is_symlink(): raise AssertionError(f"Symlink not permitted: {p.relative_to(AGENT_DIR)}")
    for name in files:
        p=Path(base)/name
        if p.suffix.lower() not in ALLOWED_SUBMISSION_EXTENSIONS:
            raise AssertionError(f"Disallowed file extension: {p.relative_to(AGENT_DIR)}")
        unpacked_bytes+=p.stat().st_size
if unpacked_bytes>MAX_SUBMISSION_SIZE_BYTES: raise AssertionError("Unpacked submission exceeds 3 GiB.")
zip_path=WORK/"submission.zip"
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(str(WORK/"submission"),"zip",root_dir=AGENT_DIR)
with zipfile.ZipFile(zip_path) as zf:
    bad=zf.testzip()
    if bad: raise RuntimeError(f"ZIP CRC failure: {bad}")
    names=set(zf.namelist())
    if "agent.yaml" not in names: raise AssertionError("agent.yaml must be at ZIP root.")
    if sum(x.file_size for x in zf.infolist())>MAX_SUBMISSION_SIZE_BYTES:
        raise AssertionError("Archive exceeds unpacked size limit.")
print(f"VALID SUBMISSION {zip_path}; model={MODEL_NAME}; files={len(names)}; unpacked={unpacked_bytes/2**20:.2f} MiB")

## 4. Graph preflight and official Gemma 4 inference server

This server is for local evaluation. The competition starts its own server during scoring. The configuration uses Gemma's tool/reasoning parsers, 4-way tensor parallelism, and LiteLLM's OpenAI-compatible model registry.

In [ ]:
import litellm,torch
from adk_submission import VllmConfig,VllmServer,discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params=True
declared=discover_declared_models(AGENT_DIR)
if declared!={MODEL_NAME}: raise AssertionError(f"Expected one model {MODEL_NAME}; found {declared}")
MODEL_PATH=Path(f"/kaggle/input/models/google/gemma-4/other/{MODEL_NAME}/2")
if not MODEL_PATH.exists(): raise FileNotFoundError(f"Attach model input: {MODEL_PATH}")
adapters=discover_adapters(str(AGENT_DIR),adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map=getattr(adapters,"adapters",None) or {}
ranks=[a.rank for a in adapter_map.values() if getattr(a,"rank",None)]
gpu_count=torch.cuda.device_count() if torch.cuda.is_available() else 0
if gpu_count<4: raise RuntimeError(f"Expected 4 L4 GPUs, detected {gpu_count}.")
extra={}
if MODEL_NAME=="gemma-4-12b-it-qat-w4a16-ct":
    extra={"--hf-overrides",'{"architectures":["Gemma4ForCausalLM"],"model_type":"gemma4"}'}
if "server_instance" in globals() and server_instance is not None:
    server_instance.stop()
vllm_cfg=VllmConfig(
    model=str(MODEL_PATH),host="127.0.0.1",port=8000,tensor_parallel_size=4,
    tool_call_parser="gemma4",reasoning_parser="gemma4",max_model_len=32768,
    dtype="bfloat16" if torch.cuda.is_bf16_supported() else "auto",
    gpu_memory_utilization=.90,enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),max_loras=max(1,len(adapter_map)),
    max_lora_rank=max(ranks) if ranks else 128,scheduler_reserve_full_isl=False,
    startup_timeout=1200,extra_args=extra,
)
server_instance=VllmServer(vllm_cfg,adapter_manifest=adapters)
server_instance.start()
models=server_instance.create_model_registry(
    aliases=[MODEL_NAME],model_prefix="openai/",api_key=os.environ.get("LOCAL_API_KEY","EMPTY"))
print(f"vLLM ready: {server_instance.base_url}; TP=4; adapters={list(adapter_map)}")

## 5. Full public development-set evaluation with resumable checkpoints

Default `EVAL_LIMIT=0` runs every loaded public task (normally 129). Set `EVAL_LIMIT=8` for a smoke check. Results are appended and fsynced after every task. The local wall-clock stop defaults to 11 hours from the first cell, leaving safety margin beneath the competition's 12-hour aggregate generation limit. `max_time_minutes` below is a **per-task** cap, not the global limit.

In [ ]:
import asyncio,concurrent.futures,pandas as pd,yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator

def run_sync(fn,*args,**kwargs):
    async def invoke():
        result=fn(*args,**kwargs)
        return await result if asyncio.iscoroutine(result) else result
    try: active=asyncio.get_running_loop().is_running()
    except RuntimeError: active=False
    if active:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda:asyncio.run(invoke())).result()
    return asyncio.run(invoke())

raw=yaml.safe_load((AGENT_DIR/"eval_config.yaml").read_text())["evaluation"]
EVAL_LIMIT=int(os.environ.get("EVAL_LIMIT","0"))
EVAL_BUDGET_HOURS=float(os.environ.get("EVAL_BUDGET_HOURS","11.0"))
if EVAL_LIMIT<0 or not 0<EVAL_BUDGET_HOURS<12: raise ValueError("Invalid evaluation budget.")
selected=tasks[:EVAL_LIMIT] if EVAL_LIMIT else tasks

fingerprint=hashlib.sha256()
for p in sorted(x for x in AGENT_DIR.rglob("*") if x.is_file()):
    fingerprint.update(p.relative_to(AGENT_DIR).as_posix().encode())
    fingerprint.update(p.read_bytes())
RESULTS=WORK/f"public_eval_{fingerprint.hexdigest()[:12]}"
RESULTS.mkdir(parents=True,exist_ok=True)
HARNESS_RESULTS=RESULTS/"harness_artifacts"; HARNESS_RESULTS.mkdir(exist_ok=True)
RESULTS_JSONL=RESULTS/"task_results.jsonl"
completed={}
if RESULTS_JSONL.is_file():
    for line in RESULTS_JSONL.read_text(encoding="utf-8").splitlines():
        if line.strip():
            row=json.loads(line); completed[row["id"]]=row
selected_ids={t.instance_id for t in selected}
completed={k:v for k,v in completed.items() if k in selected_ids}
pending=[t for t in selected if t.instance_id not in completed]
print(f"Target={len(selected)}; completed={len(completed)}; pending={len(pending)}")

eval_cfg=EvalConfig(
    tasks_path=TASKS_PATH,snapshots_dir=DATA_DIR/"snapshots",
    results_dir=HARNESS_RESULTS,submission_dir=AGENT_DIR,models=models,
    sandbox="subprocess",timeout_seconds=int(raw.get("timeout_seconds",300)),
    verification_timeout_seconds=int(raw.get("verification_timeout_seconds",300)),
    max_time_minutes=float(raw.get("max_time_minutes",5)),
    max_tool_calls=int(raw.get("max_tool_calls",50)),max_turns=int(raw.get("max_turns",60)),
    limits=limits,generation_constraints=generation_constraints,adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048,ttl_seconds=1800,cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,overlap_size=2,token_threshold=14336,event_retention_size=5),
    graph_dir=str(DATA_DIR/"graphs"),embeddings_dir=str(DATA_DIR/"embeddings"),
    wheels_dir=DATA_DIR/"wheels",verbose=False,
)
evaluator=Evaluator(eval_cfg)
with RESULTS_JSONL.open("a",encoding="utf-8") as out:
    for task in pending:
        if time.monotonic()-SESSION_STARTED>=EVAL_BUDGET_HOURS*3600:
            print(f"Stopped at local safety budget {EVAL_BUDGET_HOURS:.1f}h; resume from checkpoint.")
            break
        idx=len(completed)+1
        print(f"[{idx}/{len(selected)}] {task.instance_id} ({task.repo})")
        result=run_sync(evaluator.evaluate_task,task=task,task_index=idx,total_tasks=len(selected))
        patch=(result.agent_patch or "").strip()
        row={"id":task.instance_id,"repo":task.repo,"prediction":patch if patch else "NO_PATCH",
             "resolved":bool(result.resolved),"patch_chars":len(patch),
             "tool_calls":int(result.tool_calls),"duration_seconds":float(result.duration_seconds),
             "test_exit_code":result.test_exit_code}
        out.write(json.dumps(row,ensure_ascii=False)+"\n"); out.flush(); os.fsync(out.fileno())
        completed[task.instance_id]=row
        print(f" resolved={row['resolved']} patch={len(patch)} chars calls={row['tool_calls']} time={row['duration_seconds']:.0f}s")

rows=list(completed.values())
resolved=sum(bool(x["resolved"]) for x in rows)
summary={"model":MODEL_NAME,"evaluated":len(rows),"target":len(selected),"resolved":resolved,
         "resolution_rate":resolved/len(rows) if rows else 0.0,"complete":len(rows)==len(selected),
         "empty_patches":sum(not x["patch_chars"] for x in rows),
         "median_tool_calls":float(pd.Series([x["tool_calls"] for x in rows]).median()) if rows else None,
         "median_duration_seconds":float(pd.Series([x["duration_seconds"] for x in rows]).median()) if rows else None}
summary["per_repo"]={}
for repo in sorted({x["repo"] for x in rows}):
    subset=[x for x in rows if x["repo"]==repo]
    summary["per_repo"][repo]={"evaluated":len(subset),"resolved":sum(bool(x["resolved"]) for x in subset)}
(RESULTS/"summary.json").write_text(json.dumps(summary,indent=2))
display(pd.DataFrame(rows).tail(20))
print(json.dumps(summary,indent=2))
if summary["complete"] and resolved==len(selected):
    print("PUBLIC SET: 1.0 OBSERVED. Hidden private tasks remain unverified.")
elif summary["complete"]:
    print("PUBLIC SET below 1.0; inspect task_results.jsonl and harness logs.")
else:
    print("PUBLIC RUN incomplete; do not report its partial rate as a final benchmark score.")
print(f"Checkpoint: {RESULTS_JSONL}")
server_instance.stop()

## Engineering guardrails and next iteration

- This notebook configures the host's actual agentic harness; the host remains responsible for sandbox permissions, tool budgets, context compaction/cache, model retries, patch extraction and Phase-2 verification.
- Do not confuse per-task `max_time_minutes` with the global 12-hour budget. The notebook's local timer starts before wheel installation and inference-server setup.
- Do not train on synthetic tool traces. The supplied draft's trajectory builder fabricates tool outputs and successful tests; that creates false supervision. Train only from genuine recorded trajectories, and retain a LoRA only after cross-repository validation. The supplied GRPO example is a research sketch, not a production RL trainer.
- A full 129-task score is the first meaningful local result. It is still not a guarantee of hidden private-task success. Use per-task results and harness traces to diagnose localization, patch, timeout and regression failures.